# Análise Exploratória - Sobreviventes do Titanic

## Validando Hipótese #1

### **Analisando Variáveis Categórias**

<img src="https://media1.giphy.com/media/JcFUHp7b9mnj5a01AN/giphy.gif" width="500"/>


### Objetivo de Negócio

> Identificar fatores que levaram a alguns passageiros a sobrevirerem do Titanic e outros não. 
>
> `Motivo`: aumentar a segurança em próximos cruzeiros.

### Objetivo do Notebook

> 1. Avaliar Variáveis Categóricas (Contagem)
> 1. Visualizar Dados Categóricos (Gráfico Pizza e Barra)
> 1. Validar Hipótese (Intuição)
> 1. Validar Hipótese Estatísticamente (p-value)


### 🗺️ Hipóteses Mapeadas

> 🔵: Hipóteses tratadas neste notebook

Tem maior probabilidade de sobreviver os passageiros:
1. De classes mais altas (`Pclass` 1 > 2 > 3) 🔵 
2. Com tickets mais caros (`Fare`)
3. Com títulos de renome (`Name`)
4. Entre 20-35 anos de idade (`Age`)
5. De sexo feminino (`Sex`)
6. Com famílias pequenas (`Parch` & `SibSp`)

Bônus:
7. Família maiores são de classes mais baixas
8. Classes mais altas tem passageiros mais velhos

![Hipóteses](https://c.tenor.com/Nq-PcKRJUFsAAAAC/bart-simpson-reading.gif)



### Instala dependências externas

In [ ]:
!pip install --force-reinstall git+https://github.com/murilobellatini/eda

### Importa Libs

In [ ]:
# libs customizadas
from src.paths import RAW_DATA_PATH_TITANIC, PROCESSED_DATA_PATH_TITANIC
from src.io import load_df_from_url, mount_gdrive

# libs de visualização
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# libs de estatística
from scipy.stats import ttest_ind
import numpy as np

sns.set()

### Carrega DataFrame

In [ ]:
# montar Google Drive e carregar `raw/train.csv`
mount_gdrive()
df = pd.read_csv(f'{RAW_DATA_PATH_TITANIC}/train.csv', index_col=0)
df

### 💓 Distribuição de Sobreviventes (Geral)

> Quantos passageiros sobreviveram? E em relação ao total?

In [ ]:
# contar quantidade de sobreviventes
survived = df.Survived.value_counts() # contagem para variáveis categóricas
survived

In [ ]:
# mostrar proporcação se sobreviventes
survived_norm = df.Survived.value_counts(normalize=True) * 100
survived_norm

In [ ]:
# juntar numa tabela só
pd.concat([
           survived,
           survived_norm.rename('%')
], axis=1)

# 💡 Sacada Técnica!!

> Sempre observe o `output` do seu código.
>
> Cada transformação que fizer deve ser validada.
>
> Principalmente, quando você estiver iniciando.
>
> `Motivo`
>
> * Você vai entender o que está acontecendo....
> * E aprender mais rápido! 🤓

<img src="https://c.tenor.com/9fJnzNoBiWUAAAAd/can-you-see-it-now-close.gif" width=500/>

> *Vou ler cada linha do meu `output`...*

In [ ]:
# avaliar outras estatísticas de sobreviventes
df.Survived.describe()

In [ ]:
# renomear índices da Série
survived_norm = survived_norm.rename({0: 'Não Sobreviveu', 1: 'Sobreviveu'})
survived_norm

In [ ]:
# visualizar distribuição de sobreviventes
_ = (survived_norm
        .plot(
            kind='bar', table=True, legend=True, 
            title='Distribuição de Sobreviventes Macro',
            ylabel='% Todos Passageiros',
            fontsize=12, xticks=(),
            figsize=(7,7)
      )
    )

# 💡 Sacada Técnica!!

> Seus gráficos devem **sempre** ser auto-explicativos!
>
> Ou seja, quem ler tem que ser capaz de entender tudo sozinho
>
> `Motivo`
>
> 1. Você vai gastar menos tempo explicando
> 2. Você vai entender melhor seus dados no processo
> 3. Seu gráfico pode ser lido SEM você
>
> `Como conseguir?`
>
> Incluir **sempre**:
>
> 1. Título
> 2. Legenda
> 3. Nome dos eixos
> 4. Escala clara

<img src="https://i.pinimg.com/originals/c0/7f/3f/c07f3fe2f907344587c79af10d1c375f.gif" width=500/>

> *Fiz meu trampo, tchau...*

In [ ]:
# visualizar como gráfico pizza
_ = survived_norm.plot(
    kind='pie', table=True, legend=True,
    title='Distribuição de Sobreviventes Macro',
    figsize=(7,7)
    )

# 💡 Sacada Técnica!!

> O gráfico pizza (`pie chart`) é mal visto no ramo acadêmico e na comunidade de dados.
> 
> `Motivo`: Difícil de ler com muitas classes.
>
> `Solução`: Gráfico barra.

<img src="https://i.imgur.com/riXZoZS.gif" width=500/>

# Validação da Hipótese #1
> `Hipótese`: Tem maior probabilidade de sobreviver os passageiros de classes mais altas (`Pclass` 1 > 2 > 3)

In [ ]:
#  mostrar média de Survived
df.Survived.mean()

# 💡 Sacada Técnica!!

> A média de uma coluna booleana (0 ou 1) é o percentual de casos positivos.

In [ ]:
# mostrar média de `Survived` por `Pclass`
survived_pclass = df.groupby('Pclass').Survived.mean().rename('Survived_perc')
survived_pclass 

In [ ]:
# visualizar média de `Survived` por `Pclass`
_ = (100*survived_pclass).plot(
    kind='bar', table=True, legend=True,
    title='Distribuição de Sobreviventes por Classe',
    ylabel='% Sobreviventes', xlabel='\n\nClasse do Passageiro',
    fontsize=14, xticks=(),
    figsize=(7,7)
    )

# 👌 Insight / Conclusão

Hipótese #1 validada. ✔️

> Quanto maior a classe do passageiro, maior a probabilidade de ele ter sobrevidido.

<img src="https://media2.giphy.com/media/BfrOzZpsjfuiQ/200.gif" width=500>

> *Eu sou muito rico.*

# Calma lá! Será isso mesmo?

Mesmo sendo bem visual a diferença de sobreviventes, podemos estar errados.

Como seria possível comprovar que realmente existe diferença entre as amostras?

Afinal, a diferença pode estar dentro do erro...

<img src="https://y.yarn.co/400c0671-6c05-4010-bd16-ed990f3bdb11_text.gif" width=500>

> Não tão rápido, Sally!


## **Bônus: Como provar estatísticamente?**

Vamos falar do `p-value`, ou `valor de Pearson`.

É uma forma amplamente usada no meio acadêmico.

Serve para comparar distribuições (amostras de dados).

Se devolver um valor muito pequeno, então comprovamos que duas amostras sao diferentes.

Caso contrário, são da mesma distribuição (ou fonte de dados).

***Muito pequeno = 0.05 ou 5% (convenção)***

<img src="https://c.tenor.com/cZ_8ynUgFzAAAAAM/frink-professor-frink.gif" width=500>

> Este é o teste de Hipótese Nula para os mais nerds. 🤓



In [ ]:
# criar método para comparar distribuições com `p-value`
def compare2groups(arr1, arr2, sample_size=150, alpha=0.05):
  """
  Compara se dois grupos de dados tem origem da mesma distribuição.
  Método: Pearson-value (p-value)
  """
  arr1_sampled = np.random.choice(arr1, sample_size)
  arr2_sampled = np.random.choice(arr2, sample_size)
  stat, p = ttest_ind(arr1_sampled, arr2_sampled)
  
  print(f't-statistic: {stat}\np-value: {p}')

  if p > alpha:
    print('Same Distribution (null hypothesis not rejected)')
  else:
    print('Different Distribution (null hypothesis rejected)')

In [ ]:
# compara duas amostras de sobreviventes da 1ª Classe
compare2groups(
    df[df.Pclass==1].Survived,
    df[df.Pclass==1].Survived)

# 💡 Sacada Técnica!!

> 🧠 Sanity Check é sempre bem vindo.
>
> `Motivo`
>
> > Garantir implementação correta de funções.
>
> `Como?`
> >
> > Método validado, duas amostras da mesma distribuição PRECISAM devolver um `p-value` alto (maior que 5%).
> >
> > Ou seja, o teste confirma que as amostras tem origem da mesma distribuição.
> >
> > Isto era esperado!
> >
> > A implementação deve estar correta.

<img src="https://media1.giphy.com/media/z23hGvopHu7w4/giphy.gif" width=500>

> *Ufa...*

In [ ]:
# compara  amostras de sobreviventes da 1ª e 2ª Classe
compare2groups(
    df[df.Pclass==1].Survived,
    df[df.Pclass==2].Survived)

In [ ]:
# compara  amostras de sobreviventes da 2ª e 3ª Classe
compare2groups(
    df[df.Pclass==2].Survived,
    df[df.Pclass==3].Survived)

# 👌 Insight / Conclusão

Hipótese #1 confirmada com validade ESTATÍSTICA. ✔️

<img src="https://media0.giphy.com/media/l4q8cJzGdR9J8w3hS/giphy.gif" width=500>


## Resumo

Nesta aula aprendemos:

1. Contar variáveis categórias
2. Visualizar dados em formato barra (`bar chart`)
3. Visualizar dados em formato pizza (`pie chart`)
4. Como validar nossa 1ª hipótese na prática
5. Bonus: Como validar com rigor estatístico (p-value)

<img src="https://media1.giphy.com/media/3ov9k01Y5IKizNmC7S/200.gif" width="500"/>


# Pronto!

Pra essa aula é só...

<img src="https://c.tenor.com/IAUvvrUY7zQAAAAM/done-spongebob.gif" width="500"/>

> *Feito*